In [2]:
import pandas as pd

In [3]:
df=pd.read_csv(r"C:\Users\Kannan\Desktop\QR_MODEL1_SE_PROJECT\Model2_direct_links\merged_urls_dataset.csv")

In [4]:
df.sample(10)

,url,label
1100381,http://donkeypuncher.blogspot.fr^,adult
1091697,"http://domain-suffix,pornhub.org",adult
1766936,http://japanese-sex-movies.blogspot.fr^,adult
1428827,http://gay-facials-xxx.com,adult
556987,http://barbyestripergirl.blogspot.si^,adult
1584940,http://hellolittleflamingo.blogspot.se^,adult
3310522,http://www.maturesexmature.com^,adult
381303,http://allseelanv.blogspot.ae^,adult
1082371,http://dmrme.tumblr.com^,adult
557807,http://barebackrtxxx.com,adult


In [5]:
df.isnull().sum()

url      0
label    0
dtype: int64

In [6]:
df["label"].value_counts()

label
adult      3369166
benign      428058
phish        50535
malware      39700
Name: count, dtype: int64

In [7]:
df.shape

(3887459, 2)

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
df.columns

Index(['url', 'label'], dtype='object')

In [10]:
sampled_df = df.sample(10000, random_state=42)
x = sampled_df["url"]
y = sampled_df["label"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
x_train, x_test, y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=32)

In [12]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(x_test.shape)

(7000,)
(3000,)
(7000,)
(3000,)


In [13]:
from sklearn.preprocessing import LabelEncoder

In [14]:
encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

In [15]:
import numpy as np

In [16]:
np.unique(y_train)

array([0, 1, 2, 3])

In [17]:
np.unique(y_test)

array([0, 1, 2, 3])

In [18]:
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer

c:\Users\Kannan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
model_name="google/mobilebert-uncased"

In [20]:
tokenizer=AutoTokenizer.from_pretrained(model_name)

In [21]:
x_train.tolist()[:10]

['http://monterrey-gay.blogspot.hu^',
 'http://money4fetish.com',
 'http://sexformachos.blogspot.com',
 'http://stormyglenn.blogspot.ae^',
 'http://blackchabinexxx.blogspot.mx^',
 'http://jessicalynette.com/',
 'http://adult-escort-service-blog.blogspot.ba^',
 'http://rockzsmith.blogspot.it^',
 'http://gay-bare-plaisirs.blogspot.hu^',
 'http://0.0.0.0    www.desixxx.pro']

In [22]:
y_train

array([0, 0, 0, ..., 0, 0, 0])

In [23]:
x_train_tokenized=tokenizer(
    x_train.tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="tf"
)

c:\Users\Kannan\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [24]:
x_test_tokenized=tokenizer(
    x_test.tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="tf"
)

In [25]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "google/mobilebert-uncased", 
    num_labels=4
)

All model checkpoint layers were used when initializing TFMobileBertForSequenceClassification.

Some layers of TFMobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
# from tensorflow.keras import models,layers
import tensorflow as tf

In [27]:
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=['accuracy']
)


In [28]:
x_train_dict = { 
    'input_ids': tf.convert_to_tensor(x_train_tokenized['input_ids']),
    'attention_mask': tf.convert_to_tensor(x_train_tokenized['attention_mask'])
}

x_test_dict = {
    'input_ids': tf.convert_to_tensor(x_test_tokenized['input_ids']),
    'attention_mask': tf.convert_to_tensor(x_test_tokenized['attention_mask'])
}

model.fit(
    x=x_train_dict,
    y=y_train,
    validation_data=(x_test_dict, y_test),
    batch_size=100,
    epochs=3
)


Epoch 1/3


70/70 [==============================] - 965s 13s/step - loss: 1.0616 - accuracy: 0.8690 - val_loss: 1.0639 - val_accuracy: 0.8663
Epoch 2/3
70/70 [==============================] - 929s 13s/step - loss: 1.0616 - accuracy: 0.8690 - val_loss: 1.0639 - val_accuracy: 0.8663
Epoch 3/3
70/70 [==============================] - 946s 14s/step - loss: 1.0616 - accuracy: 0.8690 - val_loss: 1.0639 - val_accuracy: 0.8663


In [31]:
from sklearn.metrics import confusion_matrix, classification_report

In [34]:
y_pred=model.predict(x_test_tokenized)

94/94 [==============================] - 217s 2s/step


In [37]:
y_test.shape

(3000,)

In [42]:
y_pred.logits.shape

(3000, 4)

In [43]:
y_pred_classes = tf.argmax(y_pred.logits, axis=-1).numpy()

In [44]:
target_names = ["Class 0", "Class 1", "Class 2", "Class 3"]

print(classification_report(y_test, y_pred_classes, target_names=target_names))

              precision    recall  f1-score   support

     Class 0       0.87      1.00      0.93      2599
     Class 1       0.00      0.00      0.00       332
     Class 2       0.00      0.00      0.00        26
     Class 3       0.00      0.00      0.00        43

    accuracy                           0.87      3000
   macro avg       0.22      0.25      0.23      3000
weighted avg       0.75      0.87      0.80      3000



c:\Users\Kannan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Kannan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Kannan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [45]:
save_directory = "./saved_mobilebert_model"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

('./saved_mobilebert_model\\tokenizer_config.json',
 './saved_mobilebert_model\\special_tokens_map.json',
 './saved_mobilebert_model\\vocab.txt',
 './saved_mobilebert_model\\added_tokens.json',
 './saved_mobilebert_model\\tokenizer.json')